# Week 1-2: Ranking Signal Analysis & Search Foundations
**FlyRank Applied Search Intelligence ML Internship**

### Objective:
1. Understand the foundational search ranking signals across 30,000 anonymized enterprise content assets.
2. Pressure-test common industry dogmas (e.g. *Third-Party Search Volume drives organic impression capture*, *Article length determines ranking trajectory*).
3. Measure empirical Click-Through Rate (CTR) decay by position tier.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load dataset
df = pd.read_csv('../data/content_refresh_anonymized.csv')
print(f"Loaded {df.shape[0]:,} rows and {df.shape[1]} columns across {df['client_id'].nunique()} pseudonymized clients.")
df.head(3)


## 1. Myth-Busting: Search Volume vs. Actual Impressions
In classical SEO, third-party search volume metrics from tools like Ahrefs/Semrush are heavily relied upon. Let's compute the empirical Pearson and Spearman correlations with actual Google Search Console 90-day impressions.


In [ ]:
corr_pearson = df[['search_volume', 'impressions_90d']].corr().iloc[0, 1]
corr_spearman = df[['search_volume', 'impressions_90d']].corr(method='spearman').iloc[0, 1]
print(f"Pearson Correlation (Search Volume vs 90d Impressions): {corr_pearson:.5f}")
print(f"Spearman Rank Correlation: {corr_spearman:.5f}")

# Visualizing the lack of linear dependence
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=df.sample(2000, random_state=42),
    x='search_volume', y='impressions_90d',
    alpha=0.4, color='#2563eb'
)
plt.xscale('log')
plt.yscale('log')
plt.title(f'Search Volume vs Impressions (Pearson r = {corr_pearson:.4f})', fontsize=12, fontweight='bold')
plt.xlabel('Third-Party Search Volume (log scale)')
plt.ylabel('Actual GSC Impressions 90d (log scale)')
plt.grid(True, alpha=0.3)
plt.show()


## 2. Empirical CTR Decay Across Ranking Tiers
Let's analyze how Click-Through Rate degrades as average rank moves from top 3 to deep pages.


In [ ]:
ctr_summary = df.groupby('position_tier').agg(
    mean_ctr=('ctr', 'mean'),
    median_ctr=('ctr', 'median'),
    total_impressions=('impressions_90d', 'sum'),
    total_clicks=('clicks_90d', 'sum')
)
ctr_summary['weighted_ctr'] = (ctr_summary['total_clicks'] / ctr_summary['total_impressions']) * 100
tier_order = ['top_3', 'striking', 'page_1', 'page_2', 'page_3_5', 'deep']
ctr_summary = ctr_summary.reindex([t for t in tier_order if t in ctr_summary.index])
print(ctr_summary[['mean_ctr', 'weighted_ctr']])

plt.figure(figsize=(8, 4.5))
bars = plt.bar(ctr_summary.index, ctr_summary['mean_ctr'], color='#0ea5e9', edgecolor='#0284c7')
plt.title('Empirical CTR Decay Across Position Tiers', fontsize=12, fontweight='bold')
plt.xlabel('Position Tier')
plt.ylabel('Mean Observed CTR (%)')
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.02, f"{yval:.2f}%", ha='center', va='bottom', fontsize=9)
plt.grid(True, axis='y', alpha=0.3)
plt.show()


## 3. Myth-Busting: Does Word Count Determine Performance Direction?
Let's examine whether growing vs. declining content exhibits distinct word count distributions.


In [ ]:
growing_words = df[df['trend_direction'] == 'up']['word_count']
declining_words = df[df['trend_direction'] == 'down']['word_count']

print(f"Growing Content Median Word Count: {growing_words.median():.1f} words (IQR: {growing_words.quantile(0.25):.0f} - {growing_words.quantile(0.75):.0f})")
print(f"Declining Content Median Word Count: {declining_words.median():.1f} words (IQR: {declining_words.quantile(0.25):.0f} - {declining_words.quantile(0.75):.0f})")

plt.figure(figsize=(7, 4.5))
sns.boxplot(
    data=df[df['trend_direction'].isin(['up', 'down', 'stable'])],
    x='trend_direction', y='word_count',
    hue='trend_direction',
    palette={'up': '#10b981', 'down': '#ef4444', 'stable': '#64748b'},
    showfliers=False
)
plt.title('Word Count Distribution Across Performance Trajectories', fontsize=12, fontweight='bold')
plt.xlabel('Ranking Direction')
plt.ylabel('Word Count')
plt.show()


### Week 1-2 Summary & Key Takeaways:
1. **Third-Party Volume is Uncorrelated with Actual Impressions (r ≈ 0.0012)**: Keyword search volume does not capture long-tail query fanout or SERP layout dynamics.
2. **Steep CTR Degradation**: Top 3 capture exponentially more CTR than striking distance (positions 8–20) and page 2.
3. **Word Count is Non-Differentiating**: Median length is nearly identical between growing (~2,848 words) and declining (~2,909 words) pages. Optimization must focus on ranking decay velocity, intent match, and striking position recapture rather than arbitrary length inflation.
